# 10 — Análisis comparativo EEUU vs España

Núcleo del TFM: compara los dos corpus clasificados para evaluar la
transferibilidad de los patrones de fraude entre contextos.

## Entradas
- EEUU: `scam_us_FINAL_classified.csv` (1.928 tweets, clasificación BART v1)
- España: `scam_es_FINAL_classified_corregido.csv` (1.229 tweets, BART + post-corrección)

## Qué produce
1. Distribución de categorías comparada (tabla + gráfico).
2. Divergencias entre contextos.
3. Comparación de confianza.
4. Resumen de tópicos BERTopic de cada corpus.
5. CSV de la tabla comparativa, para las visualizaciones.

Sin modelos: solo análisis. Ejecución de segundos.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

us = pd.read_csv("../data/raw/scam_us_FINAL_classified.csv")
es = pd.read_csv("../data/raw/scam_es_FINAL_classified_corregido.csv")

print(f"Corpus EEUU:    {len(us)} tweets")
print(f"Corpus España:  {len(es)} tweets")


## 1. Distribución de categorías comparada

In [ ]:
us_dist = us["predicted_category"].value_counts()
es_dist = es["predicted_category"].value_counts()

cats = sorted(set(us_dist.index) | set(es_dist.index))

comp = pd.DataFrame({
    "categoria": cats,
    "EEUU_n": [int(us_dist.get(c, 0)) for c in cats],
    "ES_n":   [int(es_dist.get(c, 0)) for c in cats],
})
comp["EEUU_pct"] = (comp["EEUU_n"] / len(us) * 100).round(1)
comp["ES_pct"]   = (comp["ES_n"]   / len(es) * 100).round(1)
comp["dif_pct"]  = (comp["ES_pct"] - comp["EEUU_pct"]).round(1)
comp = comp.sort_values("EEUU_n", ascending=False).reset_index(drop=True)

print("=== DISTRIBUCIÓN COMPARADA (ordenada por frecuencia en EEUU) ===\n")
print(comp[["categoria","EEUU_n","EEUU_pct","ES_n","ES_pct","dif_pct"]].to_string(index=False))


## 2. Divergencias entre contextos

`dif_pct` positivo = más frecuente en España; negativo = más en EEUU.


In [ ]:
div = comp.sort_values("dif_pct").reset_index(drop=True)
print("=== CATEGORÍAS MÁS SOBRE-REPRESENTADAS EN EEUU (vs España) ===\n")
print(div.head(5)[["categoria","EEUU_pct","ES_pct","dif_pct"]].to_string(index=False))
print()
print("=== CATEGORÍAS MÁS SOBRE-REPRESENTADAS EN ESPAÑA (vs EEUU) ===\n")
print(div.tail(5)[["categoria","EEUU_pct","ES_pct","dif_pct"]].to_string(index=False))


## 3. Comparación de confianza

In [ ]:
print("=== CONFIANZA DE LA CLASIFICACIÓN ===\n")
for nombre, d in [("EEUU", us), ("España", es)]:
    print(f"{nombre}:")
    print(f"  Media:   {d['confidence_score'].mean():.3f}")
    print(f"  Mediana: {d['confidence_score'].median():.3f}")
    print(f"  Alta (>=0.7): {(d['confidence_score']>=0.7).sum()} ({(d['confidence_score']>=0.7).mean()*100:.1f}%)")
    print()


## 4. Tópicos BERTopic de cada corpus

In [ ]:
for nombre, d in [("EEUU", us), ("España", es)]:
    print(f"=== TÓPICOS BERTOPIC — {nombre} ===")
    if "bertopic_id" in d.columns:
        n_top = d["bertopic_id"].nunique() - (1 if -1 in d["bertopic_id"].values else 0)
        n_out = (d["bertopic_id"] == -1).sum()
        print(f"  Tópicos: {n_top} | Outliers: {n_out}")
        # top 5 tópicos por tamaño
        top = d[d["bertopic_id"]!=-1]["bertopic_id"].value_counts().head(5)
        for tid, n in top.items():
            kw = d[d["bertopic_id"]==tid]["bertopic_keywords"].iloc[0] if "bertopic_keywords" in d.columns else ""
            print(f"    Tópico {tid} ({n} tweets): {str(kw)[:70]}")
    print()


## 5. Resumen interpretativo y guardado

In [ ]:
print("="*60)
print("   RESUMEN COMPARATIVO")
print("="*60)
top_us = comp.nlargest(1,"EEUU_n").iloc[0]
top_es = comp.nlargest(1,"ES_n").iloc[0]
print(f"Categoría dominante EEUU:   {top_us['categoria']} ({top_us['EEUU_pct']}%)")
print(f"Categoría dominante España: {top_es['categoria']} ({top_es['ES_pct']}%)")
print()
print("Mayor divergencia:")
mayor = div.iloc[0]
menor = div.iloc[-1]
print(f"  {menor['categoria']}: +{menor['dif_pct']} puntos en España")
print(f"  {mayor['categoria']}: {mayor['dif_pct']} puntos (más en EEUU)")
print("="*60)

# Guardar la tabla comparativa para las visualizaciones
comp.to_csv("../data/processed/comparativa_us_es.csv", index=False, encoding="utf-8")
print()
print("✓ Guardado: data/processed/comparativa_us_es.csv")
print("  (esta tabla es la base para las visualizaciones)")
